In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.multimodal.pipeline.pipeline_text_guided_pvd import MultiModalPipelineTextGuidedPVD

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
VAL_ROOT = PROJECT_ROOT / "data" / "processed" / "PlantDocSplited_depth_AUG" / "validation"

CKPT_PATH = PROJECT_ROOT / "archive" / "multimodal_text_guided" / "multimodal_text_guided_pvd_classifier_state_dict.pth"

NUM_CLASSES = 28
BATCH_SIZE = 16
IMG_SIZE = 224
NUM_WORKERS = 0

SAVE_DIR = PROJECT_ROOT / "archive" / "multimodal_text_guided" / "validation_results"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = SAVE_DIR / "classification_report.csv"
CM_PATH = SAVE_DIR / "confusion_matrix.csv"
PRED_PATH = SAVE_DIR / "predictions.csv"

In [4]:
def class_name_to_text(class_name: str) -> str:
    clean_name = class_name.replace("_", " ").strip()
    return f"a photo of {clean_name}"

In [ ]:
class ValidationMultimodalDataset(Dataset):
    def __init__(self, root: Path, transform=None):
        self.base_dataset = datasets.ImageFolder(root=str(root), transform=transform)
        self.classes = self.base_dataset.classes
        self.class_to_idx = self.base_dataset.class_to_idx
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image_path, label = self.base_dataset.samples[idx]

        try:
            image = self.base_dataset.loader(image_path)
        except Exception as e:
            print(f"[ERROR] Failed to open validation image: {image_path}")
            raise e

        if self.transform is not None:
            image = self.transform(image)

        class_name = self.idx_to_class[label]
        text = class_name_to_text(class_name)

        return {
            "image": image,
            "text": text,
            "label": label,
            "image_path": image_path,
            "class_name": class_name,
        }


def collate_fn(batch):
    images = torch.stack([item["image"] for item in batch], dim=0)
    texts = [item["text"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    image_paths = [item["image_path"] for item in batch]
    class_names = [item["class_name"] for item in batch]

    return {
        "image": images,
        "text": texts,
        "label": labels,
        "image_path": image_paths,
        "class_name": class_names,
    }

In [8]:
from tqdm.auto import tqdm
from torchvision import transforms, datasets
from torch.utils.data import Dataset

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

class ValidationMultimodalDataset(Dataset):
    def __init__(self, root: Path, transform=None):
        self.base_dataset = datasets.ImageFolder(root=str(root), transform=transform)
        self.classes = self.base_dataset.classes
        self.class_to_idx = self.base_dataset.class_to_idx
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image_path, label = self.base_dataset.samples[idx]

        try:
            image = self.base_dataset.loader(image_path)
        except Exception as e:
            print(f"[ERROR] Failed to open validation image: {image_path}")
            raise e

        if self.transform is not None:
            image = self.transform(image)

        class_name = self.idx_to_class[label]
        text = class_name_to_text(class_name)

        return {
            "image": image,
            "text": text,
            "label": label,
            "image_path": image_path,
            "class_name": class_name,
        }

dataset = ValidationMultimodalDataset(root=VAL_ROOT, transform=transform)

print("Validation samples:", len(dataset))

bad_files = []
good_count = 0

for i in tqdm(range(len(dataset)), desc="Scanning validation dataset"):
    try:
        _ = dataset[i]
        good_count += 1
    except Exception as e:
        image_path, label = dataset.base_dataset.samples[i]
        bad_files.append({
            "idx": i,
            "image_path": image_path,
            "error": str(e),
        })
        print(f"[BAD] idx={i} path={image_path} -> {e}")

print("\nScan finished.")
print("Total good files:", good_count)
print("Total bad files:", len(bad_files))

if len(bad_files) > 0:
    print("\nBad files list:")
    for row in bad_files:
        print(row)

Validation samples: 333


Scanning validation dataset: 100%|██████████| 333/333 [03:32<00:00,  1.57it/s]


Scan finished.
Total good files: 333
Total bad files: 0


In [9]:
def main():
    print("Using device:", device)
    print("VAL_ROOT:", VAL_ROOT)
    print("CKPT_PATH:", CKPT_PATH)

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    dataset = ValidationMultimodalDataset(root=VAL_ROOT, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,   # QUAN TRỌNG
        collate_fn=collate_fn,
        pin_memory=(device == "cuda"),
    )

    print("Number of validation samples:", len(dataset))
    print("Classes:", dataset.classes)

    model = MultiModalPipelineTextGuidedPVD(
        ckpt_path=CKPT_PATH,
        num_classes=NUM_CLASSES,
        clip_model_name="ViT-L-14",
        clip_pretrained="openai",
        text_input_dim=768,
        image_input_dim=1024,
        proj_dim=512,
        proj_hidden_dim=768,
        pvd_hidden_dim=768,
        cls_hidden_dim=1024,
        dropout=0.3,
        normalize_projection=False,
        device=device,
    ).to(device)

    model.eval()

    y_true = []
    y_pred = []
    prediction_rows = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            try:
                images = batch["image"].to(device, non_blocking=True)
                texts = batch["text"]
                labels = batch["label"].to(device, non_blocking=True)
            except Exception as e:
                print(f"[ERROR] Failed to move batch {batch_idx} to device: {e}")
                raise e

            logits = model(images, texts)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(labels.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

            for i in range(len(texts)):
                true_idx = labels[i].item()
                pred_idx = preds[i].item()

                prediction_rows.append({
                    "image_path": batch["image_path"][i],
                    "text": texts[i],
                    "true_label_idx": true_idx,
                    "pred_label_idx": pred_idx,
                    "true_class_name": dataset.idx_to_class[true_idx],
                    "pred_class_name": dataset.idx_to_class[pred_idx],
                    "correct": int(true_idx == pred_idx),
                })

    acc = accuracy_score(y_true, y_pred)
    print(f"Validation Accuracy: {acc:.4f}")

    report = classification_report(
        y_true,
        y_pred,
        target_names=dataset.classes,
        digits=4,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(y_true, y_pred)

    report_df = pd.DataFrame(report).transpose()
    cm_df = pd.DataFrame(cm, index=dataset.classes, columns=dataset.classes)
    pred_df = pd.DataFrame(prediction_rows)

    report_df.to_csv(REPORT_PATH, encoding="utf-8-sig")
    cm_df.to_csv(CM_PATH, encoding="utf-8-sig")
    pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

    print("Saved classification report to:", REPORT_PATH)
    print("Saved confusion matrix to:", CM_PATH)
    print("Saved predictions to:", PRED_PATH)

if __name__ == "__main__":
    main()

Using device: cpu
VAL_ROOT: G:\My Drive\Documents\GR1\MulCo-PlantNet\data\processed\PlantDocSplited_depth_AUG\validation
CKPT_PATH: G:\My Drive\Documents\GR1\MulCo-PlantNet\archive\multimodal_text_guided\multimodal_text_guided_pvd_classifier_state_dict.pth
Number of validation samples: 333
Classes: ['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot', 'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_leaf_late_blight', 'Tomato_leaf_mosaic_virus', 'Tomato_leaf_yellow_virus', 'Tomato_mold_leaf', 'Tomato_two_spotted_spider_mites_leaf', 'grape_leaf', 'grape_leaf_black_rot']
Validation Accuracy: 0.2553
Saved classification report to: G:\My Drive\